In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: Not available")


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
import zipfile
import os

zip_path = "/content/tft.zip"
extract_path = "/content/tft"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("TFT files:")
for file in sorted(os.listdir(extract_path)):
    print(" -", file)

TFT files:
 - sequence_generation_multivariate_report.json
 - sequence_index.csv.gz
 - sequence_index_summary.csv
 - sequences_test.npz
 - sequences_train.npz
 - sequences_validation.npz


In [3]:
import numpy as np

tft_dir = "/content/tft"

train_data = np.load(
    f"{tft_dir}/sequences_train.npz",
    mmap_mode="r"
)

val_data = np.load(
    f"{tft_dir}/sequences_validation.npz",
    mmap_mode="r"
)

test_data = np.load(
    f"{tft_dir}/sequences_test.npz",
    mmap_mode="r"
)

print("Train arrays:", train_data.files)
print("Validation arrays:", val_data.files)
print("Test arrays:", test_data.files)

Train arrays: ['past_x', 'future_x', 'y', 'y_raw', 'past_feature_names', 'future_feature_names', 'store_nbr', 'item_nbr', 'family', 'class', 'perishable', 'city', 'state', 'type', 'cluster', 'input_start_date', 'input_end_date', 'target_start_date', 'target_end_date']
Validation arrays: ['past_x', 'future_x', 'y', 'y_raw', 'past_feature_names', 'future_feature_names', 'store_nbr', 'item_nbr', 'family', 'class', 'perishable', 'city', 'state', 'type', 'cluster', 'input_start_date', 'input_end_date', 'target_start_date', 'target_end_date']
Test arrays: ['past_x', 'future_x', 'y', 'y_raw', 'past_feature_names', 'future_feature_names', 'store_nbr', 'item_nbr', 'family', 'class', 'perishable', 'city', 'state', 'type', 'cluster', 'input_start_date', 'input_end_date', 'target_start_date', 'target_end_date']


In [4]:
print("TRAIN")
print("past_x :", train_data["past_x"].shape)
print("future_x:", train_data["future_x"].shape)
print("y      :", train_data["y"].shape)
print("y_raw  :", train_data["y_raw"].shape)

print("\nVALIDATION")
print("past_x :", val_data["past_x"].shape)
print("future_x:", val_data["future_x"].shape)
print("y      :", val_data["y"].shape)
print("y_raw  :", val_data["y_raw"].shape)

print("\nTEST")
print("past_x :", test_data["past_x"].shape)
print("future_x:", test_data["future_x"].shape)
print("y      :", test_data["y"].shape)
print("y_raw  :", test_data["y_raw"].shape)

print("\nSTATIC METADATA")
static_cols = [
    "store_nbr", "item_nbr", "family", "class",
    "perishable", "city", "state", "type", "cluster"
]

for col in static_cols:
    print(f"{col:12s}:", train_data[col].shape)

TRAIN
past_x : (624451, 28, 18)
future_x: (624451, 7, 14)
y      : (624451, 7)
y_raw  : (624451, 7)

VALIDATION
past_x : (3299, 28, 18)
future_x: (3299, 7, 14)
y      : (3299, 7)
y_raw  : (3299, 7)

TEST
past_x : (3237, 28, 18)
future_x: (3237, 7, 14)
y      : (3237, 7)
y_raw  : (3237, 7)

STATIC METADATA
store_nbr   : (624451,)
item_nbr    : (624451,)
family      : (624451,)
class       : (624451,)
perishable  : (624451,)
city        : (624451,)
state       : (624451,)
type        : (624451,)
cluster     : (624451,)


In [5]:
print("PAST FEATURES (18)")
for i, name in enumerate(train_data["past_feature_names"]):
    print(i, ":", name)

print("\nFUTURE FEATURES (14)")
for i, name in enumerate(train_data["future_feature_names"]):
    print(i, ":", name)

PAST FEATURES (18)
0 : unit_sales_scaled
1 : onpromotion
2 : promotion_missing
3 : was_observed
4 : day_of_week_sin
5 : day_of_week_cos
6 : month_sin
7 : month_cos
8 : day_of_month_sin
9 : day_of_month_cos
10 : week_of_year_sin
11 : week_of_year_cos
12 : time_idx
13 : is_holiday
14 : is_event
15 : is_additional
16 : is_work_day
17 : is_transfer

FUTURE FEATURES (14)
0 : day_of_week_sin
1 : day_of_week_cos
2 : month_sin
3 : month_cos
4 : day_of_month_sin
5 : day_of_month_cos
6 : week_of_year_sin
7 : week_of_year_cos
8 : time_idx
9 : is_holiday
10 : is_event
11 : is_additional
12 : is_work_day
13 : is_transfer


In [7]:
static_cols = [
    "store_nbr", "item_nbr", "family", "class",
    "perishable", "city", "state", "type", "cluster"
]

for col in static_cols:
    values = train_data[col]

    print(f"\n{col}")
    print("  dtype :", values.dtype)
    print("  unique:", len(np.unique(values)))

    # Show a few actual values
    unique_values = np.unique(values)
    print("  examples:", unique_values[:10])


store_nbr
  dtype : int64
  unique: 53
  examples: [ 1  2  3  4  5  6  7  8  9 10]

item_nbr
  dtype : int64
  unique: 122
  examples: [114800 159156 164647 167437 168927 168930 208384 215352 215468 250782]

family
  dtype : <U26
  unique: 32
  examples: ['AUTOMOTIVE' 'BABY CARE' 'BEAUTY' 'BEVERAGES' 'BREAD/BAKERY'
 'CELEBRATION' 'CLEANING' 'DAIRY' 'DELI' 'EGGS']

class
  dtype : float64
  unique: 65
  examples: [1004. 1014. 1016. 1072. 1080. 1120. 1124. 1144. 1236. 1318.]

perishable
  dtype : float64
  unique: 2
  examples: [0. 1.]

city
  dtype : <U13
  unique: 22
  examples: ['Ambato' 'Babahoyo' 'Cayambe' 'Cuenca' 'Daule' 'El Carmen' 'Esmeraldas'
 'Guaranda' 'Guayaquil' 'Ibarra']

state
  dtype : <U30
  unique: 16
  examples: ['Azuay' 'Bolivar' 'Chimborazo' 'Cotopaxi' 'El Oro' 'Esmeraldas' 'Guayas'
 'Imbabura' 'Loja' 'Los Rios']

type
  dtype : <U1
  unique: 5
  examples: ['A' 'B' 'C' 'D' 'E']

cluster
  dtype : float64
  unique: 17
  examples: [ 1.  2.  3.  4.  5.  6.  7.  8.  9.

In [8]:
# Verify static metadata consistency

static_cols = [
    "store_nbr", "item_nbr", "family", "class",
    "perishable", "city", "state", "type", "cluster"
]

print("Checking static metadata consistency...")

for col in static_cols:
    values = train_data[col]

    # Number of unique values per sequence
    unique_count = len(np.unique(values))

    print(f"{col:12s} | total unique values: {unique_count}")

print("\nStatic metadata check complete.")

Checking static metadata consistency...
store_nbr    | total unique values: 53
item_nbr     | total unique values: 122
family       | total unique values: 32
class        | total unique values: 65
perishable   | total unique values: 2
city         | total unique values: 22
state        | total unique values: 16
type         | total unique values: 5
cluster      | total unique values: 17

Static metadata check complete.


In [9]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

static_cols = [
    "store_nbr", "item_nbr", "family", "class",
    "perishable", "city", "state", "type", "cluster"
]

# Create category mappings from TRAINING data only
category_maps = {}

for col in static_cols:
    unique_values = np.unique(train_data[col])
    category_maps[col] = {
        value: idx
        for idx, value in enumerate(unique_values)
    }

print("Category mappings created:")

for col in static_cols:
    print(f"{col:12s}: {len(category_maps[col])} categories")

Category mappings created:
store_nbr   : 53 categories
item_nbr    : 122 categories
family      : 32 categories
class       : 65 categories
perishable  : 2 categories
city        : 22 categories
state       : 16 categories
type        : 5 categories
cluster     : 17 categories


In [10]:
class TFTDataset(Dataset):
    def __init__(self, data, category_maps):
        self.data = data
        self.category_maps = category_maps

        self.past_x = data["past_x"]
        self.future_x = data["future_x"]
        self.y = data["y"]
        self.y_raw = data["y_raw"]

        self.static_data = {
            col: data[col]
            for col in static_cols
        }

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        # Continuous temporal features
        past_x = torch.tensor(
            self.past_x[idx],
            dtype=torch.float32
        )

        future_x = torch.tensor(
            self.future_x[idx],
            dtype=torch.float32
        )

        y = torch.tensor(
            self.y[idx],
            dtype=torch.float32
        )

        y_raw = torch.tensor(
            self.y_raw[idx],
            dtype=torch.float32
        )

        # Convert static categorical values to integer IDs
        static_ids = []

        for col in static_cols:
            value = self.static_data[col][idx]
            static_ids.append(
                self.category_maps[col][value]
            )

        static_x = torch.tensor(
            static_ids,
            dtype=torch.long
        )

        return {
            "past_x": past_x,
            "future_x": future_x,
            "static_x": static_x,
            "y": y,
            "y_raw": y_raw
        }


# Create datasets
train_dataset = TFTDataset(train_data, category_maps)
val_dataset = TFTDataset(val_data, category_maps)
test_dataset = TFTDataset(test_data, category_maps)

# Create DataLoaders
batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Train samples: 624451
Validation samples: 3299
Test samples: 3237


In [11]:
batch = next(iter(train_loader))

print("past_x   :", batch["past_x"].shape, batch["past_x"].dtype)
print("future_x :", batch["future_x"].shape, batch["future_x"].dtype)
print("static_x :", batch["static_x"].shape, batch["static_x"].dtype)
print("y        :", batch["y"].shape, batch["y"].dtype)
print("y_raw    :", batch["y_raw"].shape, batch["y_raw"].dtype)

past_x   : torch.Size([64, 28, 18]) torch.float32
future_x : torch.Size([64, 7, 14]) torch.float32
static_x : torch.Size([64, 9]) torch.int64
y        : torch.Size([64, 7]) torch.float32
y_raw    : torch.Size([64, 7]) torch.float32


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# -----------------------------
# Gated Residual Network
# -----------------------------
class GatedResidualNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, output_size=None, dropout=0.1):
        super().__init__()

        if output_size is None:
            output_size = hidden_size

        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)

        self.gate = nn.Linear(output_size, output_size)
        self.skip = (
            nn.Linear(input_size, output_size)
            if input_size != output_size
            else nn.Identity()
        )

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(output_size)

    def forward(self, x):
        residual = self.skip(x)

        x = F.elu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        gate = torch.sigmoid(self.gate(x))
        x = gate * x + (1 - gate) * residual

        return self.norm(x)


# -----------------------------
# Variable Selection Network
# -----------------------------
class VariableSelectionNetwork(nn.Module):
    def __init__(
        self,
        num_variables,
        input_size,
        hidden_size,
        dropout=0.1
    ):
        super().__init__()

        self.num_variables = num_variables

        # One GRN for each variable
        self.variable_grns = nn.ModuleList([
            GatedResidualNetwork(
                input_size=input_size,
                hidden_size=hidden_size,
                output_size=hidden_size,
                dropout=dropout
            )
            for _ in range(num_variables)
        ])

        # Produces importance weights for variables
        self.selection_grn = GatedResidualNetwork(
            input_size=num_variables * input_size,
            hidden_size=hidden_size,
            output_size=num_variables,
            dropout=dropout
        )

    def forward(self, x):
        # x: [B, T, V, input_size]

        B, T, V, E = x.shape

        variable_outputs = []

        for i in range(V):
            variable_outputs.append(
                self.variable_grns[i](x[:, :, i, :])
            )

        variable_outputs = torch.stack(
            variable_outputs,
            dim=2
        )
        # [B, T, V, hidden_size]

        flattened = x.reshape(B, T, V * E)

        weights = self.selection_grn(flattened)
        weights = torch.softmax(weights, dim=-1)

        weights = weights.unsqueeze(-1)

        selected = torch.sum(
            weights * variable_outputs,
            dim=2
        )
        # [B, T, hidden_size]

        return selected, weights.squeeze(-1)


# -----------------------------
# Static Variable Selection
# -----------------------------
class StaticVariableSelectionNetwork(nn.Module):
    def __init__(
        self,
        num_variables,
        embedding_size,
        hidden_size,
        dropout=0.1
    ):
        super().__init__()

        self.num_variables = num_variables

        self.variable_grns = nn.ModuleList([
            GatedResidualNetwork(
                input_size=embedding_size,
                hidden_size=hidden_size,
                output_size=hidden_size,
                dropout=dropout
            )
            for _ in range(num_variables)
        ])

        self.selection_grn = GatedResidualNetwork(
            input_size=num_variables * embedding_size,
            hidden_size=hidden_size,
            output_size=num_variables,
            dropout=dropout
        )

    def forward(self, x):
        # x: [B, V, embedding_size]

        B, V, E = x.shape

        outputs = []

        for i in range(V):
            outputs.append(
                self.variable_grns[i](x[:, i, :])
            )

        outputs = torch.stack(outputs, dim=1)
        # [B, V, hidden_size]

        flattened = x.reshape(B, V * E)

        weights = self.selection_grn(flattened)
        weights = torch.softmax(weights, dim=-1)

        selected = torch.sum(
            weights.unsqueeze(-1) * outputs,
            dim=1
        )
        # [B, hidden_size]

        return selected, weights


# -----------------------------
# TFT Model
# -----------------------------
class TFT(nn.Module):
    def __init__(
        self,
        past_features=18,
        future_features=14,
        static_cardinalities=None,
        hidden_size=64,
        embedding_size=16,
        num_heads=4,
        dropout=0.1,
        forecast_horizon=7
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.embedding_size = embedding_size
        self.forecast_horizon = forecast_horizon

        # -------------------------
        # Static categorical embeddings
        # -------------------------
        if static_cardinalities is None:
            raise ValueError("static_cardinalities must be provided")

        self.static_embeddings = nn.ModuleList([
            nn.Embedding(cardinality, embedding_size)
            for cardinality in static_cardinalities
        ])

        # Static variable selection
        self.static_vsn = StaticVariableSelectionNetwork(
            num_variables=len(static_cardinalities),
            embedding_size=embedding_size,
            hidden_size=hidden_size,
            dropout=dropout
        )

        # -------------------------
        # Temporal variable embeddings
        # -------------------------
        self.past_variable_embeddings = nn.ModuleList([
            nn.Linear(1, hidden_size)
            for _ in range(past_features)
        ])

        self.future_variable_embeddings = nn.ModuleList([
            nn.Linear(1, hidden_size)
            for _ in range(future_features)
        ])

        # Variable selection networks
        self.past_vsn = VariableSelectionNetwork(
            num_variables=past_features,
            input_size=hidden_size,
            hidden_size=hidden_size,
            dropout=dropout
        )

        self.future_vsn = VariableSelectionNetwork(
            num_variables=future_features,
            input_size=hidden_size,
            hidden_size=hidden_size,
            dropout=dropout
        )

        # -------------------------
        # LSTM encoder + decoder
        # -------------------------
        self.encoder_lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.decoder_lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        # -------------------------
        # Static enrichment
        # -------------------------
        self.static_enrichment = GatedResidualNetwork(
            input_size=hidden_size * 2,
            hidden_size=hidden_size,
            output_size=hidden_size,
            dropout=dropout
        )

        # -------------------------
        # Multi-head temporal attention
        # -------------------------
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.attention_norm = nn.LayerNorm(hidden_size)

        # -------------------------
        # Final gated processing
        # -------------------------
        self.final_grn = GatedResidualNetwork(
            input_size=hidden_size,
            hidden_size=hidden_size,
            output_size=hidden_size,
            dropout=dropout
        )

        # -------------------------
        # Forecasting head
        # -------------------------
        self.forecast_head = nn.Linear(
            hidden_size,
            forecast_horizon
        )

    def embed_static(self, static_x):
        embeddings = []

        for i, embedding in enumerate(self.static_embeddings):
            embeddings.append(
                embedding(static_x[:, i])
            )

        return torch.stack(embeddings, dim=1)

    def embed_temporal(self, x, embeddings):
        outputs = []

        for i, embedding in enumerate(embeddings):
            outputs.append(
                embedding(x[:, :, i:i+1])
            )

        return torch.stack(outputs, dim=2)

    def forward(self, past_x, future_x, static_x):

        # -------------------------
        # Static processing
        # -------------------------
        static_embeddings = self.embed_static(static_x)

        static_context, static_weights = self.static_vsn(
            static_embeddings
        )

        # -------------------------
        # Past variable selection
        # -------------------------
        past_embedded = self.embed_temporal(
            past_x,
            self.past_variable_embeddings
        )

        past_selected, past_weights = self.past_vsn(
            past_embedded
        )

        # -------------------------
        # Future variable selection
        # -------------------------
        future_embedded = self.embed_temporal(
            future_x,
            self.future_variable_embeddings
        )

        future_selected, future_weights = self.future_vsn(
            future_embedded
        )

        # -------------------------
        # Encoder
        # -------------------------
        encoder_output, (h, c) = self.encoder_lstm(
            past_selected
        )

        # -------------------------
        # Decoder
        # -------------------------
        decoder_output, _ = self.decoder_lstm(
            future_selected,
            (h, c)
        )

        # -------------------------
        # Static enrichment
        # -------------------------
        static_repeated = static_context.unsqueeze(1).expand(
            -1,
            decoder_output.size(1),
            -1
        )

        enriched_input = torch.cat(
            [decoder_output, static_repeated],
            dim=-1
        )

        enriched = self.static_enrichment(
            enriched_input
        )

        # -------------------------
        # Temporal attention
        # -------------------------
        attention_output, attention_weights = self.attention(
            enriched,
            enriched,
            enriched
        )

        attention_output = self.attention_norm(
            attention_output + enriched
        )

        # -------------------------
        # Final gated processing
        # -------------------------
        final_output = self.final_grn(
            attention_output
        )

        # Use all 7 future positions
        output = self.forecast_head(
            final_output[:, -1, :]
        )

        return output

In [14]:
# Create TFT model

static_cardinalities = [
    53,   # store_nbr
    122,  # item_nbr
    32,   # family
    65,   # class
    2,    # perishable
    22,   # city
    16,   # state
    5,    # type
    17    # cluster
]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = TFT(
    past_features=18,
    future_features=14,
    static_cardinalities=static_cardinalities,
    hidden_size=64,
    embedding_size=16,
    num_heads=4,
    dropout=0.1,
    forecast_horizon=7
).to(device)

# Count parameters
num_parameters = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Device:", device)
print("Trainable parameters:", num_parameters)

Device: cuda
Trainable parameters: 808349


In [15]:
# Test one batch through the TFT

model.eval()

batch = next(iter(train_loader))

past_x = batch["past_x"].to(device)
future_x = batch["future_x"].to(device)
static_x = batch["static_x"].to(device)

with torch.no_grad():
    output = model(
        past_x,
        future_x,
        static_x
    )

print("Past input   :", past_x.shape)
print("Future input :", future_x.shape)
print("Static input :", static_x.shape)
print("Model output :", output.shape)
print("Output dtype :", output.dtype)

Past input   : torch.Size([64, 28, 18])
Future input : torch.Size([64, 7, 14])
Static input : torch.Size([64, 9])
Model output : torch.Size([64, 7])
Output dtype : torch.float32


In [17]:
import torch.optim as optim

criterion = nn.MSELoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.00025,
    weight_decay=0.0001
)

num_epochs = 30
patience = 5

best_val_loss = float("inf")
epochs_without_improvement = 0

checkpoint_path = "/content/tft_v2_best_model.pth"

print("Optimizer: AdamW")
print("Learning rate: 0.00025")
print("Weight decay: 0.0001")
print("Maximum epochs:", num_epochs)
print("Early stopping patience:", patience)
print("Checkpoint:", checkpoint_path)

Optimizer: AdamW
Learning rate: 0.0005
Weight decay: 0.0001
Maximum epochs: 30
Early stopping patience: 5
Checkpoint: /content/tft_best_model.pth


In [18]:
import time

for epoch in range(1, num_epochs + 1):

    # -------------------------
    # Training
    # -------------------------
    model.train()
    train_loss = 0.0

    start_time = time.time()

    for batch in train_loader:

        past_x = batch["past_x"].to(device, non_blocking=True)
        future_x = batch["future_x"].to(device, non_blocking=True)
        static_x = batch["static_x"].to(device, non_blocking=True)
        y = batch["y"].to(device, non_blocking=True)

        optimizer.zero_grad()

        predictions = model(
            past_x,
            future_x,
            static_x
        )

        loss = criterion(predictions, y)

        loss.backward()

        # Prevent unstable gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_loss += loss.item() * past_x.size(0)

    train_loss /= len(train_loader.dataset)

    # -------------------------
    # Validation
    # -------------------------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():

        for batch in val_loader:

            past_x = batch["past_x"].to(device, non_blocking=True)
            future_x = batch["future_x"].to(device, non_blocking=True)
            static_x = batch["static_x"].to(device, non_blocking=True)
            y = batch["y"].to(device, non_blocking=True)

            predictions = model(
                past_x,
                future_x,
                static_x
            )

            loss = criterion(predictions, y)

            val_loss += loss.item() * past_x.size(0)

    val_loss /= len(val_loader.dataset)

    elapsed = time.time() - start_time

    # -------------------------
    # Checkpoint
    # -------------------------
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            checkpoint_path
        )

        status = "✓ best"

    else:

        epochs_without_improvement += 1

        status = (
            f"no improvement "
            f"({epochs_without_improvement}/{patience})"
        )

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {elapsed:.1f}s | "
        f"{status}"
    )

    # -------------------------
    # Early stopping
    # -------------------------
    if epochs_without_improvement >= patience:

        print("\nEarly stopping triggered.")
        break

print("\nTraining complete.")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Best model saved to: {checkpoint_path}")

Epoch 01 | Train Loss: 0.196161 | Val Loss: 0.168000 | Time: 813.5s | ✓ best
Epoch 02 | Train Loss: 0.166352 | Val Loss: 0.161500 | Time: 802.9s | ✓ best
Epoch 03 | Train Loss: 0.159549 | Val Loss: 0.159268 | Time: 817.0s | ✓ best
Epoch 04 | Train Loss: 0.155398 | Val Loss: 0.160256 | Time: 814.4s | no improvement (1/5)
Epoch 05 | Train Loss: 0.152423 | Val Loss: 0.163294 | Time: 816.9s | no improvement (2/5)
Epoch 06 | Train Loss: 0.150160 | Val Loss: 0.164261 | Time: 818.6s | no improvement (3/5)
Epoch 07 | Train Loss: 0.148126 | Val Loss: 0.160051 | Time: 816.3s | no improvement (4/5)
Epoch 08 | Train Loss: 0.146396 | Val Loss: 0.161279 | Time: 813.5s | no improvement (5/5)

Early stopping triggered.

Training complete.
Best validation loss: 0.159268
Best model saved to: /content/tft_best_model.pth


In [20]:
batch = next(iter(test_loader))

print("Number of items:", len(batch))

for i, item in enumerate(batch):
    print(
        f"Item {i}: "
        f"type={type(item)}, "
        f"shape={item.shape if hasattr(item, 'shape') else 'N/A'}"
    )

Number of items: 5
Item 0: type=<class 'str'>, shape=N/A
Item 1: type=<class 'str'>, shape=N/A
Item 2: type=<class 'str'>, shape=N/A
Item 3: type=<class 'str'>, shape=N/A
Item 4: type=<class 'str'>, shape=N/A


In [21]:
batch = next(iter(test_loader))

for i, item in enumerate(batch):
    print(f"Item {i}: {repr(item)}")

Item 0: 'past_x'
Item 1: 'future_x'
Item 2: 'static_x'
Item 3: 'y'
Item 4: 'y_raw'


In [22]:
batch = next(iter(test_loader))

print(type(batch))
print(batch.keys())

for key, value in batch.items():
    print(
        f"{key}: type={type(value)}, "
        f"shape={value.shape if hasattr(value, 'shape') else 'N/A'}"
    )

<class 'dict'>
dict_keys(['past_x', 'future_x', 'static_x', 'y', 'y_raw'])
past_x: type=<class 'torch.Tensor'>, shape=torch.Size([64, 28, 18])
future_x: type=<class 'torch.Tensor'>, shape=torch.Size([64, 7, 14])
static_x: type=<class 'torch.Tensor'>, shape=torch.Size([64, 9])
y: type=<class 'torch.Tensor'>, shape=torch.Size([64, 7])
y_raw: type=<class 'torch.Tensor'>, shape=torch.Size([64, 7])


In [24]:
import joblib

scaler = joblib.load("/content/target_sales_scaler.joblib")

print("Scaler loaded successfully.")
print(type(scaler))

Scaler loaded successfully.
<class 'dict'>


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [25]:
print(scaler.keys())

for key, value in scaler.items():
    print(f"{key}: {type(value)}")


dict_keys(['target_column', 'scaled_target_column', 'pre_transform', 'post_transform', 'inverse_transform', 'target_scaler', 'fit_split'])
target_column: <class 'str'>
scaled_target_column: <class 'str'>
pre_transform: <class 'str'>
post_transform: <class 'str'>
inverse_transform: <class 'str'>
target_scaler: <class 'sklearn.preprocessing._data.StandardScaler'>
fit_split: <class 'str'>


In [26]:
target_scaler = scaler["target_scaler"]

print("Target scaler ready.")
print("Mean:", target_scaler.mean_)
print("Scale:", target_scaler.scale_)

Target scaler ready.
Mean: [3.2521175]
Scale: [1.26941991]


In [27]:
import numpy as np
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load the best TFT checkpoint (Epoch 3)
checkpoint_path = "/content/tft_best_model.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

all_preds = []
all_true = []

with torch.no_grad():
    for batch in test_loader:
        past_x = batch["past_x"].to(device)
        future_x = batch["future_x"].to(device)
        static_x = batch["static_x"].to(device)

        # TFT prediction in scaled log-space
        pred_scaled = model(past_x, future_x, static_x)

        # Convert back:
        # scaled -> log1p(sales) -> sales
        pred_log = target_scaler.inverse_transform(
            pred_scaled.cpu().numpy().reshape(-1, 1)
        )

        pred_sales = np.expm1(
            pred_log
        ).reshape(pred_scaled.shape)

        true_sales = batch["y_raw"].cpu().numpy()

        # Sales cannot be negative
        pred_sales = np.maximum(pred_sales, 0)

        all_preds.append(pred_sales)
        all_true.append(true_sales)

preds = np.concatenate(all_preds)
true = np.concatenate(all_true)

# -------------------------
# Overall metrics
# -------------------------

mae = mean_absolute_error(
    true.flatten(),
    preds.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        true.flatten(),
        preds.flatten()
    )
)

wape = (
    np.sum(np.abs(true - preds)) /
    np.sum(np.abs(true))
) * 100

smape = (
    np.mean(
        2 * np.abs(preds - true) /
        (np.abs(true) + np.abs(preds) + 1e-8)
    )
) * 100

print("===== TFT TEST RESULTS =====")
print(f"MAE   : {mae:.4f}")
print(f"RMSE  : {rmse:.4f}")
print(f"WAPE  : {wape:.2f}%")
print(f"sMAPE : {smape:.2f}%")

# -------------------------
# Day 1 → Day 7
# -------------------------

print("\n===== DAY 1 → DAY 7 =====")

for d in range(7):

    day_true = true[:, d]
    day_pred = preds[:, d]

    day_mae = mean_absolute_error(
        day_true,
        day_pred
    )

    day_rmse = np.sqrt(
        mean_squared_error(
            day_true,
            day_pred
        )
    )

    day_wape = (
        np.sum(np.abs(day_true - day_pred)) /
        np.sum(np.abs(day_true))
    ) * 100

    print(
        f"Day {d+1}: "
        f"MAE={day_mae:.4f}, "
        f"RMSE={day_rmse:.4f}, "
        f"WAPE={day_wape:.2f}%"
    )

===== TFT TEST RESULTS =====
MAE   : 12.9214
RMSE  : 37.4479
WAPE  : 29.19%
sMAPE : 38.75%

===== DAY 1 → DAY 7 =====
Day 1: MAE=12.9314, RMSE=39.2663, WAPE=27.84%
Day 2: MAE=13.0399, RMSE=38.8156, WAPE=29.01%
Day 3: MAE=12.9703, RMSE=38.4466, WAPE=28.91%
Day 4: MAE=13.0545, RMSE=35.0369, WAPE=28.97%
Day 5: MAE=12.7831, RMSE=34.0098, WAPE=28.76%
Day 6: MAE=12.8541, RMSE=38.1190, WAPE=29.83%
Day 7: MAE=12.8162, RMSE=38.1110, WAPE=31.26%


In [28]:
from google.colab import files

files.download("/content/tft_best_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>